# Module 2: Data Cleaning and Structuring
## Objective
Clean the raw military dataset by removing unwanted symbols, standardizing column names, converting values to numeric types, handling missing values, validating the cleaned data, and producing a dataset ready for Tableau/Power BI.

**Deliverables:** `military_cleaned.csv` and `cleaning_report.txt`.

## Step 1: Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

input_file='military_raw_data.csv'
output_file='military_cleaned.csv'

if not Path(input_file).exists():
    raise FileNotFoundError(f"{input_file} not found.")

df=pd.read_csv(input_file)
rows_before, cols_before=df.shape
missing_before=int(df.isnull().sum().sum())

print("Dataset Loaded")
print("Rows:",rows_before," Columns:",cols_before)
display(df.head())

Dataset Loaded
Rows: 145  Columns: 60


,Country,Power Index,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,total_military_aircraft,...,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km,Continent,Region,GDP,Alliance
0,United States,0.0741,"341,963,408","150,463,900","124,816,644","4,445,524","1,333,030","799,500",0,"13,032",...,"495,156,000 \t\t\t\t\t\...","247,883,000,000 \t\t\t\...","9,833,517 \t\t\t\t\t\t\...","19,924 \t\t\t\t\t\t\r\n...","12,002 \t\t\t\t\t\t\r\n...","41,009 \t\t\t\t\t\t\r\n...",Americas,Northern America,2.550000e+13,NATO
1,Russia,0.0791,"140,820,810","69,002,197","46,189,226","1,267,387","1,320,000","2,000,000","250,000","4,237",...,"290,763,000 \t\t\t\t\t\...","162,166,000,000 \t\t\t\...","17,098,242 \t\t\t\t\t\t...","37,653 \t\t\t\t\t\t\r\n...","22,407 \t\t\t\t\t\t\r\n...","102,000 \t\t\t\t\t\t\r\...",Europe,Eastern Europe,NaN,NaN
2,China,0.0919,"1,415,043,270","764,123,366","626,864,169","19,810,606","2,035,000","510,000","625,000","3,529",...,"5,191,000,000 \t\t\t\t\...","157,041,000,000 \t\t\t\...","9,596,960 \t\t\t\t\t\t\...","14,500 \t\t\t\t\t\t\r\n...","22,457 \t\t\t\t\t\t\r\n...","27,700 \t\t\t\t\t\t\r\n...",Asia,Eastern Asia,1.800000e+13,NaN
3,India,0.1346,"1,409,128,296","662,290,299","522,786,598","23,955,181","1,431,000","1,000,000","2,527,000","2,183",...,"1,262,000,000 \t\t\t\t\...","127,727,000,000 \t\t\t\...","3,287,263 \t\t\t\t\t\t\...","7,000 \t\t\t\t\t\t\r\n\...","13,888 \t\t\t\t\t\t\r\n...","14,500 \t\t\t\t\t\t\r\n...",Asia,Southern Asia,3.390000e+12,NaN
4,South Korea,0.1642,"52,081,799","26,040,900","21,353,538","416,654","450,000","3,100,000","120,000","1,540",...,"136,817,000 \t\t\t\t\t\...","326,000,000 \t\t\t\t\t\...","99,720 \t\t\t\t\t\t\r\n...","2,413 \t\t\t\t\t\t\r\n\...",237 \t\t\t\t\t\t\r\n\t\...,"1,600 \t\t\t\t\t\t\r\n\...",Asia,Eastern Asia,NaN,NaN


## Step 2: Explore the Dataset

In [2]:
display(df.info())
display(df.describe(include='all'))
print("Duplicate Rows:",df.duplicated().sum())
display(df.isnull().sum().to_frame("Missing Values"))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 60 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Country                                    145 non-null    object 
 1   Power Index                                145 non-null    float64
 2   total_population                           145 non-null    object 
 3   total_military_manpower                    145 non-null    object 
 4   fit_for_service                            145 non-null    object 
 5   population_reaching_military_age_annually  145 non-null    object 
 6   active_personnel                           145 non-null    object 
 7   reserve_personnel                          145 non-null    object 
 8   paramilitary                               145 non-null    object 
 9   total_military_aircraft                    145 non-null    object 
 10  fighter_aircraft          

None

,Country,Power Index,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,total_military_aircraft,...,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km,Continent,Region,GDP,Alliance
count,145,145.000000,145,145,145,145,145,145,145,145,...,145,145,145,145,145,145,136,136,1.180000e+02,29
unique,145,NaN,145,145,145,145,126,85,58,112,...,118,79,145,108,133,94,5,14,NaN,1
top,United States,NaN,"341,963,408","150,463,900","124,816,644","4,445,524","25,000",0,0,80,...,0 \t\t\t\t\t\t\r\n\t\t\...,0 \t\t\t\t\t\t\r\n\t\t\...,"9,833,517 \t\t\t\t\t\t\...",0 \t\t\t\t\t\t\r\n\t\t\...,0 \t\t\t\t\t\t\r\n\t\t\...,0 \t\t\t\t\t\t\r\n\t\t\...,Asia,Sub-Saharan Africa,NaN,NATO
freq,1,NaN,1,1,1,1,5,52,26,4,...,26,62,1,32,10,43,42,29,NaN,29
mean,NaN,1.642772,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.792383e+11,NaN
std,NaN,1.135182,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.916658e+12,NaN
min,NaN,0.074100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.382619e+09,NaN
25%,NaN,0.651700,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.039700e+10,NaN
50%,NaN,1.543200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.202187e+10,NaN
75%,NaN,2.396900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.472500e+11,NaN


Duplicate Rows: 0


,Missing Values
Country,0
Power Index,0
total_population,0
total_military_manpower,0
fit_for_service,0
population_reaching_military_age_annually,0
active_personnel,0
reserve_personnel,0
paramilitary,0
total_military_aircraft,0


## Step 3: Standardize Column Names

In [3]:
df.columns=(df.columns.str.strip()
            .str.lower()
            .str.replace(" ","_",regex=False)
            .str.replace("-","_",regex=False)
            .str.replace(r"[^a-zA-Z0-9_]","",regex=True))
print(df.columns.tolist())

['country', 'power_index', 'total_population', 'total_military_manpower', 'fit_for_service', 'population_reaching_military_age_annually', 'active_personnel', 'reserve_personnel', 'paramilitary', 'total_military_aircraft', 'fighter_aircraft', 'attack_aircraft', 'transport_aircraft', 'trainer_aircraft', 'special_mission_aircraft', 'tanker_aircraft', 'total_military_helicopters', 'attack_helicopters', 'tanks', 'armored_fighting_vehicles', 'self_propelled_artillery', 'towed_artillery', 'rocket_projectors', 'total_naval_fleet', 'total_naval_fleet_tonnage_mt', 'aircraft_carriers', 'helicopter_carriers', 'submarines', 'destroyers', 'frigates', 'corvettes', 'coastal_patrol_craft', 'mine_warfare_craft', 'defense_budget_usd', 'external_debt_usd', 'purchasing_power_parity_usd', 'foreign_exchange_and_gold_reserves_usd', 'total_serviceable_airports', 'labour_force', 'major_ports_and_terminals', 'total_merchant_marine_fleet', 'railway_coverage_km', 'roadway_coverage_km', 'oil_production_bbl', 'oil_c

## Step 4: Clean Text Values

In [4]:
def clean_value(x):
    if pd.isna(x):
        return x
    x=str(x)
    x=x.replace("\t"," ").replace("\r"," ").replace("\n"," ")
    x=re.sub(r"[,%+$*()]", "", x)
    x=x.replace(",","")
    x=re.sub(r"\s+"," ",x).strip()
    return x

object_cols=df.select_dtypes(include='object').columns
df[object_cols]=df[object_cols].apply(lambda col: col.apply(clean_value))
print("Special symbols removed.")

Special symbols removed.


## Step 5: Convert Numeric Columns

In [5]:
converted_columns=[]
for col in df.columns:
    temp=pd.to_numeric(df[col],errors='coerce')
    if temp.notna().sum()>=len(df)*0.5:
        df[col]=temp
        converted_columns.append(col)

print("Columns converted to numeric:",len(converted_columns))
print(converted_columns)

Columns converted to numeric: 40
['power_index', 'total_population', 'total_military_manpower', 'fit_for_service', 'population_reaching_military_age_annually', 'active_personnel', 'reserve_personnel', 'paramilitary', 'total_military_aircraft', 'fighter_aircraft', 'attack_aircraft', 'transport_aircraft', 'trainer_aircraft', 'special_mission_aircraft', 'tanker_aircraft', 'total_military_helicopters', 'attack_helicopters', 'tanks', 'armored_fighting_vehicles', 'self_propelled_artillery', 'towed_artillery', 'rocket_projectors', 'total_naval_fleet', 'aircraft_carriers', 'helicopter_carriers', 'submarines', 'destroyers', 'frigates', 'corvettes', 'coastal_patrol_craft', 'mine_warfare_craft', 'defense_budget_usd', 'external_debt_usd', 'purchasing_power_parity_usd', 'foreign_exchange_and_gold_reserves_usd', 'total_serviceable_airports', 'labour_force', 'major_ports_and_terminals', 'total_merchant_marine_fleet', 'gdp']


## Step 6: Handle Missing Values

In [6]:
numeric=df.select_dtypes(include='number').columns
categorical=df.select_dtypes(exclude='number').columns

for col in numeric:
    df[col]=df[col].fillna(df[col].median())

for col in categorical:
    mode=df[col].mode()
    df[col]=df[col].fillna(mode.iloc[0] if not mode.empty else "Unknown")

duplicates_before=df.duplicated().sum()
df=df.drop_duplicates()
print("Duplicates Removed:",duplicates_before)

Duplicates Removed: 0


## Step 7: Validate the Cleaned Dataset

In [7]:
missing_after=int(df.isnull().sum().sum())
rows_after,cols_after=df.shape

print("Rows Before:",rows_before)
print("Rows After :",rows_after)
print("Columns    :",cols_after)
print("Missing Before:",missing_before)
print("Missing After :",missing_after)
print("Maximum Missing %:",((df.isnull().sum()/len(df))*100).max())

important=['power_index','total_population','active_personnel','defense_budget_usd','gdp']
print("\nValidation of Important Columns")
for c in important:
    if c in df.columns:
        print(c,":",df[c].dtype)

display(df.dtypes.to_frame("Data Type"))
display(df.describe(include='all'))

Rows Before: 145
Rows After : 145
Columns    : 60
Missing Before: 255
Missing After : 0
Maximum Missing %: 0.0

Validation of Important Columns
power_index : float64
total_population : int64
active_personnel : int64
defense_budget_usd : int64
gdp : float64


,Data Type
country,object
power_index,float64
total_population,int64
total_military_manpower,int64
fit_for_service,int64
population_reaching_military_age_annually,int64
active_personnel,int64
reserve_personnel,int64
paramilitary,int64
total_military_aircraft,int64


,country,power_index,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,total_military_aircraft,...,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km,continent,region,gdp,alliance
count,145,145.000000,1.450000e+02,1.450000e+02,1.450000e+02,1.450000e+02,1.450000e+02,1.450000e+02,1.450000e+02,145.000000,...,145,145,145,145,145,145,145,145,1.450000e+02,145
unique,145,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,118,79,145,108,133,94,5,14,NaN,1
top,United States,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0 mt,0 Cu.M,9833517 km,0 km,0 km,0 km,Asia,Sub-Saharan Africa,NaN,NATO
freq,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,26,62,1,32,10,43,51,38,NaN,145
mean,NaN,1.642772,5.456547e+07,2.581125e+07,2.019565e+07,8.546933e+05,1.624485e+05,2.371013e+05,1.218919e+05,359.041379,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.512739e+11,NaN
std,NaN,1.135182,1.702420e+08,8.583942e+07,6.928101e+07,2.667628e+06,2.967677e+05,6.517850e+05,6.095925e+05,1191.810935,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.642709e+12,NaN
min,NaN,0.074100,3.640360e+05,8.372800e+04,5.023700e+04,1.820000e+03,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.382619e+09,NaN
25%,NaN,0.651700,5.650957e+06,2.392390e+06,1.912981e+06,7.199100e+04,2.250000e+04,0.000000e+00,2.000000e+03,25.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.306889e+10,NaN
50%,NaN,1.543200,1.469705e+07,6.351857e+06,4.096295e+06,2.048300e+05,5.100000e+04,2.010000e+04,1.250000e+04,99.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.202187e+10,NaN
75%,NaN,2.396900,3.879481e+07,1.794604e+07,1.403738e+07,6.250060e+05,1.600000e+05,1.685000e+05,5.500000e+04,271.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.440000e+11,NaN


## Step 8: Save Outputs

In [8]:
df.to_csv(output_file,index=False)

with open("cleaning_report.txt","w") as f:
    f.write("MILITARY DATA CLEANING REPORT\n")
    f.write("="*45+"\n")
    f.write(f"Rows Before Cleaning : {rows_before}\n")
    f.write(f"Rows After Cleaning  : {rows_after}\n")
    f.write(f"Columns              : {cols_after}\n")
    f.write(f"Missing Before       : {missing_before}\n")
    f.write(f"Missing After        : {missing_after}\n")
    f.write(f"Converted Columns    : {len(converted_columns)}\n")
    f.write(f"Duplicates Removed   : {duplicates_before}\n")
    f.write("Status               : Cleaning Completed Successfully\n")

print("Saved:",output_file)
print("Report Generated: cleaning_report.txt")

Saved: military_cleaned.csv
Report Generated: cleaning_report.txt


## Results
- Dataset cleaned successfully.
- Missing values handled.
- Numeric fields converted.
- Duplicate rows removed.
- Output dataset generated.

## Conclusion
The dataset has been cleaned, validated, and standardized for further KPI engineering and dashboard development in Tableau or Power BI.